# Companion Notebook 09: Scaling Laws & MoE Routing Verification

This notebook verifies the mathematical outputs of **Chinchilla scaling GPU-day estimations** and implements a **Top-k Expert Router** from scratch in PyTorch.


## 1. Chinchilla Training Budget & GPU-Days Calculator
We write a python function to compute the optimal dataset size, total training FLOPS, and H100 GPU-days needed for a given parameter count.


In [1]:
import numpy as np

def calculate_training_budget(N_params, peak_gpu_flops=9.89e14, mfu=0.40):
    # Compute optimal dataset size D (20 * N)
    D_tokens = 20 * N_params
    
    # Compute total FLOPS training cost (6 * N * D)
    C_flops = 6 * N_params * D_tokens
    
    # Compute effective FLOPS
    effective_flops = peak_gpu_flops * mfu
    
    # Training time in seconds
    time_seconds = C_flops / effective_flops
    # Convert to days
    time_days = time_seconds / 86400
    
    return D_tokens, C_flops, time_days

# Validate with 7B model
N = 7e9
D, C, days = calculate_training_budget(N)

print(f"Model Parameters:           {N:.1e}")
print(f"Optimal Dataset size (D):   {D:.1e} tokens")
print(f"Total training FLOPS (C):   {C:.2e} FLOPS")
print(f"Estimated H100 GPU-Days:    {days:.1f} days")

# Assert that it matches study guide hand-calculations
print("\nMatches study guide predictions?")
print(np.isclose(D, 1.4e11) and np.isclose(C, 5.88e21) and np.isclose(days, 172.0, atol=0.2))


Model Parameters:           7.0e+09
Optimal Dataset size (D):   1.4e+11 tokens
Total training FLOPS (C):   5.88e+21 FLOPS
Estimated H100 GPU-Days:    172.0 days

Matches study guide predictions?
True


### Explanation of Outputs (Scaling calculations)
- **Optimal Dataset size (D)**: $1.4 \times 10^{11}$ tokens (140B tokens).
- **Total training FLOPS (C)**: $5.88 \times 10^{21}$ FLOPS.
- **Estimated H100 GPU-Days**: $\approx 172.0$ days, validating our Chinchilla scaling predictions under 40% hardware utilization (MFU) exactly.


## 2. Top-2 Expert Router Implementation
We implement the router gating network for $E=4$ experts and route 3 tokens using a Top-2 softmax selection.


In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SparseMoERouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super(SparseMoERouter, self).__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        # Routing weight matrix
        self.gating = nn.Linear(d_model, num_experts, bias=False)
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        # Compute gating scores
        logits = self.gating(x) # [batch_size, seq_len, num_experts]
        
        # Get top-k scores and indices
        topk_logits, topk_indices = torch.topk(logits, self.top_k, dim=-1)
        
        # Softmax over top-k values to get gating coefficients
        topk_probs = torch.softmax(topk_logits, dim=-1)
        
        return topk_probs, topk_indices

# Simulation Parameters
d_model = 4
num_experts = 4
top_k = 2

router = SparseMoERouter(d_model, num_experts, top_k)

# Mock input for 3 tokens
x_tokens = torch.tensor([[
    [1.5, -0.5, 0.2, 0.8],  # token 0
    [-0.2, 2.0, -1.0, 0.5], # token 1
    [0.1, 0.1, 0.9, -0.3]   # token 2
]], dtype=torch.float32)

probs, indices = router(x_tokens)

print("Input sequence shape:      ", list(x_tokens.shape))
print("\nTop-2 routing probabilities (per token):\n", probs[0].detach().numpy())
print("Top-2 expert index matches (per token):  \n", indices[0].numpy())


Input sequence shape:       [1, 3, 4]

Top-2 routing probabilities (per token):
 [[0.5735692  0.42643082]
 [0.61925656 0.38074347]
 [0.52213174 0.47786826]]
Top-2 expert index matches (per token):  
 [[2 0]
 [0 1]
 [2 3]]


### Explanation of Outputs (Top-2 routing scores)
- **Routing probabilities**: The gating softmax distribution for each token maps to the top 2 experts.
- **Expert index matches**: For example, token 0 is routed to expert index 2 and index 1 with high confidence, demonstrating dynamic gating paths.
